# Home Credit: preprocessing and feature engineering

All transformations are fitted on the training set only to prevent validation and test leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
df = pd.read_csv('application_train.csv')

# 365243 is a documented sentinel for unknown employment duration, not a real value.
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator

# Interpretable features available at the time of the loan application.
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
df['EMPLOYMENT_YEARS'] = -df['DAYS_EMPLOYED'] / 365.25
df['CREDIT_TO_INCOME'] = safe_divide(df['AMT_CREDIT'], df['AMT_INCOME_TOTAL'])
df['ANNUITY_TO_INCOME'] = safe_divide(df['AMT_ANNUITY'], df['AMT_INCOME_TOTAL'])
df['CREDIT_TO_ANNUITY'] = safe_divide(df['AMT_CREDIT'], df['AMT_ANNUITY'])
df['GOODS_TO_CREDIT'] = safe_divide(df['AMT_GOODS_PRICE'], df['AMT_CREDIT'])

ext_source_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
df['EXT_SOURCE_MEAN'] = df[ext_source_cols].mean(axis=1)
df['EXT_SOURCE_STD'] = df[ext_source_cols].std(axis=1)
df['EXT_SOURCE_MISSING_COUNT'] = df[ext_source_cols].isna().sum(axis=1)

document_cols = [col for col in df.columns if col.startswith('FLAG_DOCUMENT_')]
df['DOCUMENT_COUNT'] = df[document_cols].sum(axis=1)
bureau_cols = [col for col in df.columns if col.startswith('AMT_REQ_CREDIT_BUREAU_')]
df['CREDIT_BUREAU_REQUESTS_TOTAL'] = df[bureau_cols].sum(axis=1, min_count=1)

# Preserve the raw values; the pipeline below handles missingness without leaking information.
df.head()

## Exploratory data analysis

Inspect class imbalance and missingness before fitting the model.

In [ ]:
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Duplicate rows: {df.duplicated().sum():,}')

target_summary = df['TARGET'].value_counts().rename_axis('TARGET').reset_index(name='count')
target_summary['percent'] = 100 * target_summary['count'] / len(df)
display(target_summary)

missing_summary = (
    df.drop(columns='TARGET').isna().mean().mul(100)
    .sort_values(ascending=False).rename('missing_percent').to_frame()
)
display(missing_summary.head(15))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df, x='TARGET', ax=axes[0], color='#4C78A8')
axes[0].set_title('Target class distribution')
axes[0].set_xlabel('TARGET (1 = payment difficulty)')
axes[0].set_ylabel('Applications')

top_missing = missing_summary.head(15).sort_values('missing_percent')
axes[1].barh(top_missing.index, top_missing['missing_percent'], color='#F58518')
axes[1].set_title('15 features with most missing data')
axes[1].set_xlabel('Missing values (%)')
plt.tight_layout()
plt.show()

## Stratified train/validation/test split and preprocessing

Median imputation and missingness indicators are used for numeric features. Categorical missing values become their own `Missing` category.

In [ ]:
y = df['TARGET'].astype('int64')
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])

# 64% train, 16% validation, 20% untouched test; all splits retain the class ratio.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=RANDOM_STATE
)

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_cols),
        ('categorical', categorical_transformer, categorical_cols),
    ],
    remainder='drop'
)

# Fit only on training features; validation and test remain unseen until evaluation.
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

positive_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Train: {X_train.shape}; validation: {X_val.shape}; test: {X_test.shape}')
print(f'Processed feature count: {X_train_processed.shape[1]:,}')
print(f'Positive-class weight for BCEWithLogitsLoss: {positive_weight:.2f}')